# Echoes of Astra: voices with ElevenLabs v3

This notebook records every voice line in the game (about 400 clips) with your ElevenLabs account and packs them into a ZIP file. Nothing to install: everything runs in your browser on Google's servers.

**How to use it:**
1. Click ▶ next to each step, **one at a time, top to bottom**. If Colab warns that the notebook was not written by Google, choose **Run anyway**.
2. In step 1, paste your API key into the field that appears. The key stays in this session only and is never saved anywhere.
3. Step 3 designs the unique voices: listen to them. Step 4 plays a short test.
4. Step 5 records everything and takes about 20–40 minutes. If it stops, run step 5 again and it carries on from where it stopped.
5. Step 6 downloads the ZIP (sometimes several parts). Send it to Claude in the chat.
6. Step 7 (optional) removes the designed voices from your account, to free up voice slots.

**Re-recording round:** when the script lists characters to redo, the notebook records only their lines (4 takes each) and designs only their new voices. It takes a few minutes and costs few credits. In step 2 you can choose the scope: only the voices you reported, or those plus a few more characters with the same problem.


In [ ]:
#@title 1) API key and account check
import getpass, json, os, time, base64, zipfile, threading, hashlib, concurrent.futures as cf
import requests
from IPython.display import HTML, display

API = 'https://api.elevenlabs.io'
KEY = getpass.getpass('Paste your ElevenLabs API key and press Enter: ').strip()
S = requests.Session(); S.headers.update({'xi-api-key': KEY})

def call(method, path, *, retries=7, **kw):
    last = None
    for a in range(retries):
        try:
            r = S.request(method, API + path, timeout=300, **kw)
        except requests.RequestException as e:
            last = e; time.sleep(3 + 3 * a); continue
        if r.status_code in (409, 429, 500, 502, 503, 504):
            last = r; time.sleep(min(60, 4 * (a + 1))); continue
        return r
    if isinstance(last, requests.Response): return last
    raise RuntimeError(f'No connection to ElevenLabs: {last}')

def err(r):
    try:
        d = r.json(); return json.dumps(d.get('detail', d), ensure_ascii=False)[:500]
    except Exception:
        return r.text[:500]

def player(b, mime='audio/mpeg', label=''):
    display(HTML(f'<div style="margin:4px 0"><b>{label}</b><br><audio controls src="data:{mime};base64,{base64.b64encode(b).decode()}"></audio></div>'))

rv = call('GET', '/v1/voices')
if rv.status_code != 200:
    raise SystemExit(f'The key does not work ({rv.status_code}): {err(rv)}\nCheck the key and its permissions (Voices: Read/Write, Text to Speech, Text to Voice).')
VOICES = rv.json().get('voices', [])
SUB = {}
rs = call('GET', '/v1/user/subscription')
if rs.status_code == 200:
    SUB = rs.json()
    left = SUB.get('character_limit', 0) - SUB.get('character_count', 0)
    print(f"Plan: {SUB.get('tier')} · credits left this month: {left:,} of {SUB.get('character_limit', 0):,} · custom voice limit: {SUB.get('voice_limit')}")
else:
    print('No access to plan details (the key lacks "User: Read"). Assuming the Starter plan.')
print(f'Voices visible on the account: {len(VOICES)}. Everything OK, go to step 2.')


In [ ]:
#@title 2) Load the script and plan the recording
SCOPE = 'Reported voices + similar ones'  #@param ['Reported voices + similar ones', 'Only the reported voices']
BIBLE = json.loads(r"""{"designs": [{"key": "dragon", "description": "An ancient, colossal dragon. Extremely deep, resonant, gravelly male voice with enormous chest resonance; slow, regal and contemptuous, with a low growl at the ends of phrases. Dark fantasy video game villain.", "text": "Bow, insects. I have burned kingdoms older than your bloodline, and I will burn yours before the sun has set. Kneel, and perhaps I will let you watch.", "seed": 1101, "guidance": 5.0}, {"key": "demon_queen", "description": "A demon queen. Low, velvety, cold female voice with a faint smoky rasp; amused and cruel, speaking slowly with elegant menace. Dark fantasy video game villain.", "text": "Hope is such a fragile little thing. I have watched it flicker out in a thousand eyes, and every time, it was beautiful. Come closer. Let me watch yours.", "seed": 1102, "guidance": 5.0}, {"key": "demon_brute", "description": "A hulking demon warrior. Very deep, rough, snarling male voice, heavy breathing, guttural and aggressive, barely controlling his rage. Dark fantasy video game.", "text": "You fed a soul to the pit, and the pit sent me. I will tear down your walls, break your banners, and feast on every scream. Run, if it makes you feel better.", "seed": 1103, "guidance": 5.0}, {"key": "mad_scientist", "description": "An excitable, theatrical vampire scientist. Middle-aged British male voice, quick and slightly nasal, giggling between words, delighted by gruesome experiments, a little unhinged.", "text": "Ahh, fresh variables! Hold still, hold still, this will only sting for a moment. Oh, look at that reaction! Fascinating. Somebody write that down, quickly!", "seed": 1104, "guidance": 5.0}, {"key": "seraph", "description": "A serene celestial angel. Luminous, airy, warm female voice with gentle authority; calm and soothing, speaking slowly, as if from a vast cathedral.", "text": "The light descends. Rise, fallen ones, for your war is not over. Mercy burns brighter than hate, and no darkness has ever outlasted the dawn.", "seed": 1105, "guidance": 5.0}, {"key": "kobold", "description": "A tiny, scrappy lizard creature. High-pitched, raspy, eager young male voice, fast and a little squeaky, overconfident and comically brave. Fantasy video game.", "text": "I'm a dragon too! Well, a small one. Sort of! But my teeth are very sharp, and I bite really, really fast. Fear me! Please? Just a little?", "seed": 1106, "guidance": 5.0}, {"key": "combat_ai", "description": "A military combat artificial intelligence. Calm, precise, synthetic female voice, clipped and emotionless, perfectly even pacing, slightly metallic. Science fiction video game.", "text": "Vanguard unit online. All systems green. Hostile signatures detected in sector three. Engaging protocol initiated. Stand clear of the firing line.", "seed": 1107, "guidance": 5.0}, {"key": "nature_god", "description": "An ancient primal god of nature. Deep, earthy, slow, rumbling male voice, warm but immense, like wind moving through an old forest. Fantasy video game.", "text": "The wild answers my call. Come forth, children of the earth, root and claw and wing. What was taken from the forest will return to the soil.", "seed": 1108, "guidance": 5.0}, {"key": "cat_witch", "description": "A sly feline vampire witch. Playful, sultry young female voice with a purring, teasing, mischievous lilt; she enjoys toying with her prey. Fantasy video game.", "text": "Mmm, the blood moon is purring tonight, and so am I. Bleed for me, darling. Slowly, if you can. I have nine lives, and you have only the one.", "seed": 1109, "guidance": 5.0}, {"key": "abomination", "description": "A stitched-together laboratory abomination. Wet, guttural, distorted male voice, slow and labored, groaning between words, barely human. Horror video game monster.", "text": "Consume. Assimilate. Feed the machine. Stitched together, flesh and wire, we grow stronger with every piece you leave behind. Come closer.", "seed": 1110, "guidance": 5.0}, {"key": "baroness", "description": "A decadent vampire baroness. Elegant, refined, husky British female voice; languid and seductive, aristocratic, thirsty for blood.", "text": "Kneel, darlings. Your Baroness has arrived, and she is thirsty. Your blood is a fine vintage, and I intend to take the whole bottle.", "seed": 1111, "guidance": 5.0}, {"key": "bunny", "description": "A bubbly rabbit-girl brawler. Bright, energetic young female voice, cocky and playful, giggly, always excited for a fight.", "text": "Brynn's here! Okay, who wants to get smashed first? Bonk! Ha, you didn't even scratch me! Come on, come on, one more round!", "seed": 1113, "guidance": 5.0}, {"key": "samurai", "description": "A noble fox samurai. Calm, low, disciplined male voice, measured and honorable, speaking with quiet intensity.", "text": "I am Akeno. Draw your blade, and face me with honor. One cut, one truth. Cherry blossoms fall so beautifully, and so must we all.", "seed": 1114, "guidance": 5.0}, {"key": "captain", "description": "A seasoned captain of the guard. Strong, warm baritone male voice, commanding and inspiring, a born battlefield leader.", "text": "Guard, form up! I lead, you follow, and nobody falls behind. With me, strike as one, and strike true! We hold this line until the sun comes up.", "seed": 1115, "guidance": 5.0}, {"key": "merchant", "description": "A fast-talking rabbit merchant. Cheerful, slightly nasal, middle-aged male voice with salesman patter, charming and sly.", "text": "Step right up! Something for everyone, and a little extra for you, friend. No refunds, no returns, no exceptions, but oh, what bargains!", "seed": 1116, "guidance": 5.0}, {"key": "gladiator", "description": "A massive dragonkin gladiator. Booming, gravelly, showboating male voice, loud and theatrical, always playing to the crowd.", "text": "Step aside! The champion takes the arena! Let me hear you! The crowd wants blood, and I always deliver. Who is next? Who wants to be famous?", "seed": 1117, "guidance": 5.0}, {"key": "oracle", "description": "An ancient dragon oracle. Deep, wise, mystical male voice, calm and prophetic, with a soft rumble beneath every word.", "text": "I have seen this battle a thousand times, little one, and you lose in every one. Your future is already ash. I foresaw even this moment.", "seed": 1118, "guidance": 5.0}, {"key": "pontiff", "description": "An aging holy emperor and high priest. Grand, sonorous, elderly male voice, solemn and righteous, echoing authority.", "text": "Heaven opens its gates for the faithful. Kneel, and be lifted. Fall before the Direction, heretic, for no sinner escapes the eyes of heaven.", "seed": 1119, "guidance": 5.0}, {"key": "assassin", "description": "A professional assassin. Quiet, cool, breathy female voice, detached and precise, almost a whisper.", "text": "Target marked. Contract signed. Nothing personal. Quiet now, this will only take a second. You will not even hear me coming.", "seed": 1121, "guidance": 5.0}, {"key": "butler", "description": "An impeccably polite vampire butler. Old, dry, refined British male voice, calm and courteous, with dark humor.", "text": "At your service, my liege. Tea, or bloodshed? Do pardon the intrusion, sir. It has been an honor serving this house for three hundred years.", "seed": 1122, "guidance": 5.0}, {"key": "squire", "description": "An eager young squire. Light, earnest, youthful male voice, nervous but brave, cracking with excitement.", "text": "Hey, ugly! Yes, you! Look at me, not them! For my promotion, and for the banner! I've been practicing all week, teacher, just watch me!", "seed": 1124, "guidance": 5.0}, {"key": "inquisitor", "description": "A cold holy inquisitor. Sharp, controlled, icy female voice, precise and intimidating.", "text": "Confess. Your little tricks won't save you now. Heresy is punished, every single time. The Direction sees all, and it does not forgive.", "seed": 1125, "guidance": 5.0}, {"key": "firedancer", "description": "An ecstatic fire dancer. Warm, breathy, joyful female voice, wild and rhythmic, laughing with excitement.", "text": "Dance with me in the flames! Don't be shy! Feel the rhythm of fire, now spin, spin, spin! The music never stops while the embers burn.", "seed": 1126, "guidance": 5.0}, {"key": "braggart", "description": "A vain dragonkin warrior. Pompous, booming, self-admiring male voice, theatrical and arrogant.", "text": "Behold! The most magnificent warrior you will ever see. Try not to swoon. Witness my greatness, up close, and remember my name forever.", "seed": 1128, "guidance": 5.0}, {"key": "paladin2", "description": "Young holy knight woman in her late twenties. Warm, strong alto with real emotion and grit, slightly husky; passionate and fierce like the lead actress of a dark fantasy video game, never like a narrator or an audiobook reader. Natural breathing, burning conviction.", "text": "By the Concord's light, judgment comes! Kneel before justice, and stay down. Even if I fall here, the light remains, and it will find you.", "seed": 2101, "guidance": 3.5}, {"key": "zealot", "description": "Fanatical young demon cultist woman. Wild-eyed and breathless; her voice cracks into shrieks of ecstatic devotion, then drops to a feverish whisper. Raw, unhinged, theatrical villain performance, not a narrator.", "text": "For the Dominion, I give everything! Glory to Valrath, glory unending! Take my blood, take my breath, take it all, and let them burn!", "seed": 2102, "guidance": 3.5}, {"key": "outrider", "description": "Cocky, energetic young wolf-girl scout in her early twenties. Playful and teasing, laughing while she talks, a little breathless from running; bright, scrappy and fast. Spontaneous animated-series actress, not a narrator.", "text": "Last one there skips dinner! Ha! Too slow, way too slow! Come on, try to keep up, I have been running circles around you all day!", "seed": 2103, "guidance": 3.5}, {"key": "officer_f", "description": "Battle-hardened female army officer in her thirties. Firm, slightly raspy voice; shouting orders over gunfire with urgency and grit, but human and tired underneath. Gritty military video game performance, not a narrator.", "text": "Drone, on me! Squad, we move out on my mark! Advance, advance! Keep your heads down and keep pushing forward, nobody gets left behind!", "seed": 2104, "guidance": 3.5}, {"key": "researcher_m", "description": "Giddy young male vampire scientist in his twenties. Nerdy, fast-talking, excitable and a little nasal, giggling and cackling at his own terrible experiments. Comedic, energetic, animated performance, not a narrator.", "text": "Testing time! Subject Two, wake up, you are my favorite! For science! Well, mostly for science. Oh, oh, look at that reaction! Fascinating!", "seed": 2105, "guidance": 3.5}, {"key": "scion2", "description": "Young vampire mafia heiress in her twenties. Low, smoky, purring female voice with a lazy, possessive drawl and a light Italian accent; she smiles while she threatens, every word dripping with entitlement. Sultry villain performance in a dark fantasy video game, not a narrator.", "text": "What's yours is mine, darling. It always was. Drown in my shadow, and when you wake up, you'll belong to me. Umbrasco never forgets, and Umbrasco never forgives.", "seed": 2106, "guidance": 3.5}, {"key": "renata", "description": "Scrappy young woman sky pilot around twenty, a reckless thrill-seeker. Bright, slightly raspy voice, talking fast with a grin, laughing at danger and shouting over roaring engines. Energetic animated-series heroine performance, not a narrator.", "text": "Engines hot! Hold on to something, we're going up! Ha, full throttle, full throttle! Nobody flies like me, nobody! Keep flying, keep flying, and don't you dare look down!", "seed": 2107, "guidance": 3.5}, {"key": "selena", "description": "Charismatic young woman leader of a rebellion, early thirties. Warm, clear mezzo voice that swells with emotion when she rallies her people; heartfelt, urgent and human, her voice catching with feeling. Rousing heroine performance in a fantasy war game, not a narrator.", "text": "Stand with me! Not for me, for each other! Together, now, hold the line! If we fall, we fall as one, and if we rise, we rise together. Stay united, whatever happens!", "seed": 2108, "guidance": 3.5}, {"key": "valeria", "description": "Bold, cocky female dragoon knight in her late twenties, a daredevil aerial lancer. Confident, slightly husky voice with a grin in it, breathing hard from steep dives and shouting fierce taunts into the wind. Spirited action-heroine performance, not a narrator.", "text": "Valeria, diving in! Out of my sky! Ha, did you see that? Again! Dive, strike, climb, and never look back. Nobody touches these clouds while I'm still flying!", "seed": 2109, "guidance": 3.5}, {"key": "blademaiden2", "description": "Playful fox-spirit swordswoman in her twenties. Silky, teasing female voice with an amused little laugh; light on her feet, toying with her opponents like a cat with a mouse, elegant but mischievous. Animated fantasy game performance, not a narrator.", "text": "Nine tails, one blade. Shall we dance? Too slow, darling, far too slow! Every step you take, I have already taken twice. Oh, don't pout. Try again, I'm having fun.", "seed": 2110, "guidance": 3.5}], "cast": {"IMP002": {"design": null, "premade": ["Charlie", "Chris", "Eric"], "gender": "male"}, "IMP003": {"design": "cat_witch", "premade": ["Charlotte", "Jessica", "Laura"], "gender": "female"}, "IMP004": {"design": "demon_brute", "premade": ["Callum", "Brian"], "gender": "male"}, "IMP005": {"design": "demon_queen", "premade": ["Aria", "Alice"], "gender": "female"}, "IMP006": {"design": null, "premade": ["Daniel", "George"], "gender": "male"}, "IMP008": {"design": "demon_brute", "premade": ["Brian", "Callum"], "gender": "male"}, "IMP009": {"design": "dragon", "premade": ["Brian", "George"], "gender": "male"}, "IMP010": {"design": "baroness", "premade": ["Lily", "Alice"], "gender": "female"}, "IMP011": {"design": "scion2", "premade": ["Charlotte", "Aria"], "gender": "female"}, "IMP012": {"design": "assassin", "premade": ["Sarah", "River"], "gender": "female"}, "IMP013": {"design": null, "premade": ["Brian", "Roger"], "gender": "male"}, "IMP014": {"design": "researcher_m", "premade": ["Liam", "Will"], "gender": "male"}, "IMP015": {"design": "demon_queen", "premade": ["Aria", "Lily"], "gender": "female"}, "IMP016": {"design": "demon_brute", "premade": ["Daniel", "Callum"], "gender": "male"}, "IMP017": {"design": "zealot", "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP018": {"design": "butler", "premade": ["George", "Daniel"], "gender": "male"}, "IMP019": {"design": "mad_scientist", "premade": ["Callum", "George"], "gender": "male"}, "IMP033": {"design": null, "premade": ["Alice", "Matilda"], "gender": "female"}, "IMP037": {"design": "squire", "premade": ["Liam", "Will"], "gender": "male"}, "IMP039": {"design": null, "premade": ["Daniel", "Roger"], "gender": "male"}, "IMP040": {"design": "bunny", "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP041": {"design": "paladin2", "premade": ["Alice", "Matilda"], "gender": "female"}, "IMP042": {"design": null, "premade": ["Eric", "Roger"], "gender": "male"}, "IMP043": {"design": null, "premade": ["Sarah", "River"], "gender": "female"}, "IMP044": {"design": "officer_f", "premade": ["Alice", "Matilda"], "gender": "female"}, "IMP045": {"design": null, "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP046": {"design": "selena", "premade": ["Matilda", "Alice"], "gender": "female"}, "IMP047": {"design": null, "premade": ["Sarah", "Aria"], "gender": "female"}, "IMP048": {"design": "seraph", "premade": ["Lily", "Sarah"], "gender": "female"}, "IMP051": {"design": "captain", "premade": ["Roger", "Brian"], "gender": "male"}, "IMP066": {"design": "pontiff", "premade": ["George", "Bill"], "gender": "male"}, "IMP069": {"design": null, "premade": ["Callum", "Liam"], "gender": "male"}, "IMP070": {"design": "inquisitor", "premade": ["Alice", "Aria"], "gender": "female"}, "IMP071": {"design": null, "premade": ["Chris", "Roger"], "gender": "male"}, "IMP072": {"design": null, "premade": ["Lily", "Alice"], "gender": "female"}, "IMP073": {"design": "renata", "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP075": {"design": null, "premade": ["Alice", "Aria"], "gender": "female"}, "IMP077": {"design": null, "premade": ["Liam", "Will"], "gender": "male"}, "IMP078": {"design": "valeria", "premade": ["Matilda", "Laura"], "gender": "female"}, "IMP080": {"design": "merchant", "premade": ["Charlie", "Eric"], "gender": "male"}, "IMP096": {"design": null, "premade": ["Brian", "Callum"], "gender": "male"}, "IMP097": {"design": null, "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP098": {"design": null, "premade": ["Will", "Liam"], "gender": "male"}, "IMP099": {"design": "assassin", "premade": ["River", "Sarah"], "gender": "female"}, "IMP101": {"design": "firedancer", "premade": ["Jessica", "Charlotte"], "gender": "female"}, "IMP102": {"design": "oracle", "premade": ["George", "Brian"], "gender": "male"}, "IMP103": {"design": "kobold", "premade": ["Will", "Liam"], "gender": "male"}, "IMP104": {"design": "outrider", "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP105": {"design": null, "premade": ["Lily", "Matilda"], "gender": "female"}, "IMP107": {"design": null, "premade": ["Liam", "Charlie"], "gender": "male"}, "IMP108": {"design": "gladiator", "premade": ["Brian", "Roger"], "gender": "male"}, "IMP109": {"design": null, "premade": ["Laura", "Matilda"], "gender": "female"}, "IMP110": {"design": "braggart", "premade": ["Callum", "George"], "gender": "male"}, "IMP111": {"design": "dragon", "premade": ["Brian", "George"], "gender": "male"}, "IMP113": {"design": "dragon", "premade": ["Brian", "Callum"], "gender": "male"}, "IMP114": {"design": null, "premade": ["Daniel", "Roger"], "gender": "male"}, "IMP127": {"design": "samurai", "premade": ["Daniel", "Callum"], "gender": "male"}, "IMP128": {"design": null, "premade": ["River", "Sarah"], "gender": "female"}, "IMP129": {"design": null, "premade": ["George", "Daniel"], "gender": "male"}, "IMP130": {"design": "nature_god", "premade": ["Brian", "Bill"], "gender": "male"}, "IMP131": {"design": null, "premade": ["Lily", "Matilda"], "gender": "female"}, "IMP132": {"design": "demon_queen", "premade": ["River", "Sarah"], "gender": "female"}, "IMP134": {"design": "blademaiden2", "premade": ["Charlotte", "Alice"], "gender": "female"}, "IMP135": {"design": null, "premade": ["Callum", "Daniel"], "gender": "male"}, "IMP136": {"design": null, "premade": ["Lily", "River"], "gender": "female"}, "IMP138": {"design": null, "premade": ["Liam", "Eric"], "gender": "male"}, "IMP139": {"design": null, "premade": ["Matilda", "Alice"], "gender": "female"}, "IMP140": {"design": "squire", "premade": ["Will", "Liam"], "gender": "male"}, "IMP141": {"design": null, "premade": ["Bill", "Brian"], "gender": "male"}, "IMP142": {"design": null, "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP143": {"design": null, "premade": ["Jessica", "Sarah"], "gender": "female"}, "IMP144": {"design": null, "premade": ["Charlie", "Callum"], "gender": "male"}, "IMP163": {"design": null, "premade": ["Callum", "Daniel"], "gender": "male"}, "IMP160": {"design": null, "premade": ["Laura", "Matilda"], "gender": "female"}, "IMP034": {"design": "combat_ai", "premade": ["Sarah", "River"], "gender": "female"}, "IMP035": {"design": null, "premade": ["Roger", "Brian"], "gender": "male"}, "IMP036": {"design": null, "premade": ["Chris", "Eric"], "gender": "male"}, "IMP049": {"design": null, "premade": ["Brian", "Roger"], "gender": "male"}, "IMP050": {"design": "combat_ai", "premade": ["River", "Sarah"], "gender": "female"}, "IMP065": {"design": "combat_ai", "premade": ["River", "Sarah"], "gender": "female"}, "IMP067": {"design": null, "premade": ["Daniel", "George"], "gender": "male"}, "IMP068": {"design": "combat_ai", "premade": ["River", "Alice"], "gender": "female"}, "IMP076": {"design": null, "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP079": {"design": null, "premade": ["Brian", "Callum"], "gender": "male"}, "IMP081": {"design": null, "premade": ["Will", "Eric"], "gender": "male"}, "IMP159": {"design": null, "premade": ["Charlie", "Chris"], "gender": "male"}, "TOKEN_DRONE": {"design": "combat_ai", "premade": ["River", "Sarah"], "gender": "female"}, "IMP001": {"design": "abomination", "premade": ["Brian", "Callum"], "gender": "male"}, "IMP074": {"design": "abomination", "premade": ["Brian", "Callum"], "gender": "male"}, "TOKEN_EXPERIMENT": {"design": "abomination", "premade": ["Brian", "Callum"], "gender": "male"}, "SOV_M": {"design": null, "premade": ["George", "Daniel", "Roger"], "gender": "male"}, "SOV_F": {"design": null, "premade": ["Alice", "Lily", "Matilda"], "gender": "female"}, "ANN": {"design": null, "premade": ["Aria", "Alice", "Lily"], "gender": "female"}}, "jobs": [{"stem": "IMP002_play", "cast": "IMP002", "text": "[sly] Fresh stock. No questions."}, {"stem": "IMP002_attack", "cast": "IMP002", "text": "[menacing] Hold still."}, {"stem": "IMP002_death", "cast": "IMP002", "text": "[weakly] Pay my widow..."}, {"stem": "IMP003_play", "cast": "IMP003", "text": "[purring] The blood moon purrs..."}, {"stem": "IMP003_attack", "cast": "IMP003", "text": "[teasing] Bleed for me, darling."}, {"stem": "IMP003_death", "cast": "IMP003", "text": "[chuckles weakly] Eight lives left..."}, {"stem": "IMP004_play", "cast": "IMP004", "text": "[growling] The pit sent me!"}, {"stem": "IMP004_attack", "cast": "IMP004", "text": "[roaring] Tear them apart!"}, {"stem": "IMP004_death", "cast": "IMP004", "text": "[rasping] More... blood..."}, {"stem": "IMP005_play", "cast": "IMP005", "text": "[cold] The harvest begins."}, {"stem": "IMP005_attack", "cast": "IMP005", "text": "[cruel] Kneel, mortal."}, {"stem": "IMP005_death", "cast": "IMP005", "text": "[fading] Reaped..."}, {"stem": "IMP006_play", "cast": "IMP006", "text": "[commanding] Rise, my faithful!"}, {"stem": "IMP006_attack", "cast": "IMP006", "text": "[venomous] Wither away!"}, {"stem": "IMP006_death", "cast": "IMP006", "text": "[weak laugh] I'll be back..."}, {"stem": "IMP008_play", "cast": "IMP008", "text": "[menacing growl] Burn it all."}, {"stem": "IMP008_attack", "cast": "IMP008", "text": "[shouting] To ashes!"}, {"stem": "IMP008_death", "cast": "IMP008", "text": "[rasping] The fire dies..."}, {"stem": "IMP009_play", "cast": "IMP009", "text": "[menacing] No grave holds me."}, {"stem": "IMP009_attack", "cast": "IMP009", "text": "[roaring] Rot beneath me!"}, {"stem": "IMP009_death", "cast": "IMP009", "text": "[defiant] I will rise..."}, {"stem": "IMP010_play", "cast": "IMP010", "text": "[sultry] Kneel, darlings."}, {"stem": "IMP010_attack", "cast": "IMP010", "text": "[savoring] A fine vintage."}, {"stem": "IMP010_death", "cast": "IMP010", "text": "[proud] Remember my house..."}, {"stem": "IMP011_play", "cast": "IMP011", "text": "[smug, purring] What's yours is mine."}, {"stem": "IMP011_attack", "cast": "IMP011", "text": "[menacing whisper] Drown in shadow."}, {"stem": "IMP011_death", "cast": "IMP011", "text": "[bitter, pained] Umbrasco remembers..."}, {"stem": "IMP012_play", "cast": "IMP012", "text": "[quiet] Target marked."}, {"stem": "IMP012_attack", "cast": "IMP012", "text": "[whispers] Nothing personal."}, {"stem": "IMP012_death", "cast": "IMP012", "text": "[pained whisper] Compromised..."}, {"stem": "IMP013_play", "cast": "IMP013", "text": "[menacing] Syndicate business."}, {"stem": "IMP013_attack", "cast": "IMP013", "text": "[threatening] Pay up, or bleed!"}, {"stem": "IMP013_death", "cast": "IMP013", "text": "[gasping] I didn't squeal..."}, {"stem": "IMP014_play", "cast": "IMP014", "text": "[giddy, cackling] Testing time!"}, {"stem": "IMP014_attack", "cast": "IMP014", "text": "[excited] For science!"}, {"stem": "IMP014_death", "cast": "IMP014", "text": "[weakly, whining] My notes..."}, {"stem": "IMP015_play", "cast": "IMP015", "text": "[velvety menace] Abandon hope."}, {"stem": "IMP015_attack", "cast": "IMP015", "text": "[cruel] Despair, little one."}, {"stem": "IMP015_death", "cast": "IMP015", "text": "[cold laugh] Only a delay..."}, {"stem": "IMP016_play", "cast": "IMP016", "text": "[barking orders] For the Dominion!"}, {"stem": "IMP016_attack", "cast": "IMP016", "text": "[shouting] Forward, or die!"}, {"stem": "IMP016_death", "cast": "IMP016", "text": "[gasping] Traitors..."}, {"stem": "IMP017_play", "cast": "IMP017", "text": "[ecstatic, shrieking] I give everything!"}, {"stem": "IMP017_attack", "cast": "IMP017", "text": "[fanatical scream] Glory to Valrath!"}, {"stem": "IMP017_death", "cast": "IMP017", "text": "[feverish whisper] Still... serving..."}, {"stem": "IMP018_play", "cast": "IMP018", "text": "[politely] At your service."}, {"stem": "IMP018_attack", "cast": "IMP018", "text": "[courteous] Do pardon me."}, {"stem": "IMP018_death", "cast": "IMP018", "text": "[dignified] An honor, master..."}, {"stem": "IMP019_play", "cast": "IMP019", "text": "[giddy] Fresh variables!"}, {"stem": "IMP019_attack", "cast": "IMP019", "text": "[excited] Hold still, specimen!"}, {"stem": "IMP019_death", "cast": "IMP019", "text": "[fascinated] Fascinating..."}, {"stem": "IMP033_play", "cast": "IMP033", "text": "[determined] Shields up!"}, {"stem": "IMP033_attack", "cast": "IMP033", "text": "[strained effort] Push them back!"}, {"stem": "IMP033_death", "cast": "IMP033", "text": "[pained] Shield's broken..."}, {"stem": "IMP037_play", "cast": "IMP037", "text": "[shouting] Hey, ugly! Over here!"}, {"stem": "IMP037_attack", "cast": "IMP037", "text": "[excited] For the banner!"}, {"stem": "IMP037_death", "cast": "IMP037", "text": "[weak] Was I brave?"}, {"stem": "IMP039_play", "cast": "IMP039", "text": "[calm] My guard never falls."}, {"stem": "IMP039_attack", "cast": "IMP039", "text": "[fierce] Steel answers steel!"}, {"stem": "IMP039_death", "cast": "IMP039", "text": "[solemn] My oath endures..."}, {"stem": "IMP040_play", "cast": "IMP040", "text": "[cocky] Brynn's here!"}, {"stem": "IMP040_attack", "cast": "IMP040", "text": "[gleeful] Bonk! Ha!"}, {"stem": "IMP040_death", "cast": "IMP040", "text": "[pouting] No fair..."}, {"stem": "IMP041_play", "cast": "IMP041", "text": "[fierce, passionate] Judgment comes!"}, {"stem": "IMP041_attack", "cast": "IMP041", "text": "[battle cry] Kneel before justice!"}, {"stem": "IMP041_death", "cast": "IMP041", "text": "[pained, defiant] The light... remains..."}, {"stem": "IMP042_play", "cast": "IMP042", "text": "[brisk] Recalculating."}, {"stem": "IMP042_attack", "cast": "IMP042", "text": "[confident] Execute plan B!"}, {"stem": "IMP042_death", "cast": "IMP042", "text": "[surprised] Didn't plan this..."}, {"stem": "IMP043_play", "cast": "IMP043", "text": "[calm] Target acquired."}, {"stem": "IMP043_attack", "cast": "IMP043", "text": "[exhales] One shot."}, {"stem": "IMP043_death", "cast": "IMP043", "text": "[pained] Lost my sightline..."}, {"stem": "IMP044_play", "cast": "IMP044", "text": "[shouting orders] Squad, move out!"}, {"stem": "IMP044_attack", "cast": "IMP044", "text": "[battle shout] Advance!"}, {"stem": "IMP044_death", "cast": "IMP044", "text": "[gasping, strained] Keep... pushing..."}, {"stem": "IMP045_play", "cast": "IMP045", "text": "[cheerful] Medic here, nyaa!"}, {"stem": "IMP045_attack", "cast": "IMP045", "text": "[playful] Sorry, nyaa!"}, {"stem": "IMP045_death", "cast": "IMP045", "text": "[weakly] Who heals me...?"}, {"stem": "IMP046_play", "cast": "IMP046", "text": "[rousing, passionate] Stand with me!"}, {"stem": "IMP046_attack", "cast": "IMP046", "text": "[shouting] Together, now!"}, {"stem": "IMP046_death", "cast": "IMP046", "text": "[tender, weakly] Stay united..."}, {"stem": "IMP047_play", "cast": "IMP047", "text": "[steady] Holding position."}, {"stem": "IMP047_attack", "cast": "IMP047", "text": "[shouting] Opening fire!"}, {"stem": "IMP047_death", "cast": "IMP047", "text": "[pained] Trooper down!"}, {"stem": "IMP048_play", "cast": "IMP048", "text": "[serene] The light descends."}, {"stem": "IMP048_attack", "cast": "IMP048", "text": "[powerful] Be cleansed!"}, {"stem": "IMP048_death", "cast": "IMP048", "text": "[peaceful] I will return..."}, {"stem": "IMP051_play", "cast": "IMP051", "text": "[commanding] Guard, form up!"}, {"stem": "IMP051_attack", "cast": "IMP051", "text": "[battle cry] With me, Guard!"}, {"stem": "IMP051_death", "cast": "IMP051", "text": "[strained] Carry on..."}, {"stem": "IMP066_play", "cast": "IMP066", "text": "[solemn] Kneel, and be lifted."}, {"stem": "IMP066_attack", "cast": "IMP066", "text": "[righteous anger] Fall, heretic!"}, {"stem": "IMP066_death", "cast": "IMP066", "text": "[weak] Heaven... chooses another..."}, {"stem": "IMP069_play", "cast": "IMP069", "text": "[rallying shout] Riders, descend!"}, {"stem": "IMP069_attack", "cast": "IMP069", "text": "[exhilarated] Strike and soar!"}, {"stem": "IMP069_death", "cast": "IMP069", "text": "[gasping] My wings..."}, {"stem": "IMP070_play", "cast": "IMP070", "text": "[icy] Confess. Now."}, {"stem": "IMP070_attack", "cast": "IMP070", "text": "[sharp] Heresy is punished!"}, {"stem": "IMP070_death", "cast": "IMP070", "text": "[cold] The Direction sees..."}, {"stem": "IMP071_play", "cast": "IMP071", "text": "[bored] You're under lockdown."}, {"stem": "IMP071_attack", "cast": "IMP071", "text": "[sarcastic] Resisting arrest?"}, {"stem": "IMP071_death", "cast": "IMP071", "text": "[groaning] Lockdown lifted..."}, {"stem": "IMP072_play", "cast": "IMP072", "text": "[smug] Every piece in place."}, {"stem": "IMP072_attack", "cast": "IMP072", "text": "[amused] Check."}, {"stem": "IMP072_death", "cast": "IMP072", "text": "[gracious] Well played..."}, {"stem": "IMP073_play", "cast": "IMP073", "text": "[excited, laughing] Engines hot!"}, {"stem": "IMP073_attack", "cast": "IMP073", "text": "[whooping] Full throttle!"}, {"stem": "IMP073_death", "cast": "IMP073", "text": "[wistful, fading] Keep flying..."}, {"stem": "IMP075_play", "cast": "IMP075", "text": "[stern] None shall pass."}, {"stem": "IMP075_attack", "cast": "IMP075", "text": "[fierce] From above!"}, {"stem": "IMP075_death", "cast": "IMP075", "text": "[resigned] Grounded..."}, {"stem": "IMP077_play", "cast": "IMP077", "text": "[euphoric] I feel alive!"}, {"stem": "IMP077_attack", "cast": "IMP077", "text": "[frenzied] More! More!"}, {"stem": "IMP077_death", "cast": "IMP077", "text": "[trembling] One more dose..."}, {"stem": "IMP078_play", "cast": "IMP078", "text": "[bold, grinning] Valeria, diving in!"}, {"stem": "IMP078_attack", "cast": "IMP078", "text": "[fierce shout] Out of my sky!"}, {"stem": "IMP078_death", "cast": "IMP078", "text": "[pained, sad] Clipped wings..."}, {"stem": "IMP080_play", "cast": "IMP080", "text": "[salesman] Step right up!"}, {"stem": "IMP080_attack", "cast": "IMP080", "text": "[indignant] No refunds!"}, {"stem": "IMP080_death", "cast": "IMP080", "text": "[sighs] Closed for business..."}, {"stem": "IMP096_play", "cast": "IMP096", "text": "[smug] Your supplies? Ours."}, {"stem": "IMP096_attack", "cast": "IMP096", "text": "[snarling] Pay the toll!"}, {"stem": "IMP096_death", "cast": "IMP096", "text": "[growling] The pack remembers..."}, {"stem": "IMP097_play", "cast": "IMP097", "text": "[wild] Blitz! Go, go!"}, {"stem": "IMP097_attack", "cast": "IMP097", "text": "[whooping] Raid's on!"}, {"stem": "IMP097_death", "cast": "IMP097", "text": "[gasping] Raid's over..."}, {"stem": "IMP098_play", "cast": "IMP098", "text": "[sly] Smells like loot."}, {"stem": "IMP098_attack", "cast": "IMP098", "text": "[playful] Quick bite!"}, {"stem": "IMP098_death", "cast": "IMP098", "text": "[weak] Lost the trail..."}, {"stem": "IMP099_play", "cast": "IMP099", "text": "[whispers] The hunt begins."}, {"stem": "IMP099_attack", "cast": "IMP099", "text": "[cold whisper] Already gone."}, {"stem": "IMP099_death", "cast": "IMP099", "text": "[ironic] Now I'm the prey..."}, {"stem": "IMP101_play", "cast": "IMP101", "text": "[joyful] Dance with me!"}, {"stem": "IMP101_attack", "cast": "IMP101", "text": "[ecstatic] Feel the fire!"}, {"stem": "IMP101_death", "cast": "IMP101", "text": "[sad] The music stops..."}, {"stem": "IMP102_play", "cast": "IMP102", "text": "[prophetic] I have seen your end."}, {"stem": "IMP102_attack", "cast": "IMP102", "text": "[booming] Your future burns!"}, {"stem": "IMP102_death", "cast": "IMP102", "text": "[serene] Foreseen..."}, {"stem": "IMP103_play", "cast": "IMP103", "text": "[proud, squeaky] I'm a dragon too!"}, {"stem": "IMP103_attack", "cast": "IMP103", "text": "[fierce but tiny] Bite! Bite!"}, {"stem": "IMP103_death", "cast": "IMP103", "text": "[sad, squeaky] Too small..."}, {"stem": "IMP104_play", "cast": "IMP104", "text": "[laughing, cheeky] Last one skips dinner!"}, {"stem": "IMP104_attack", "cast": "IMP104", "text": "[teasing, laughing] Too slow!"}, {"stem": "IMP104_death", "cast": "IMP104", "text": "[groans, annoyed] Tripped... seriously?"}, {"stem": "IMP105_play", "cast": "IMP105", "text": "[mystical] Foretold in gold."}, {"stem": "IMP105_attack", "cast": "IMP105", "text": "[calm] As it was written."}, {"stem": "IMP105_death", "cast": "IMP105", "text": "[peaceful] Another dawn..."}, {"stem": "IMP107_play", "cast": "IMP107", "text": "[hyped] Glory or nothing!"}, {"stem": "IMP107_attack", "cast": "IMP107", "text": "[effort] Swing! Swing!"}, {"stem": "IMP107_death", "cast": "IMP107", "text": "[satisfied] Worth it..."}, {"stem": "IMP108_play", "cast": "IMP108", "text": "[booming showman] The champion is here!"}, {"stem": "IMP108_attack", "cast": "IMP108", "text": "[roaring] Blood for the crowd!"}, {"stem": "IMP108_death", "cast": "IMP108", "text": "[stunned] The crowd... silent..."}, {"stem": "IMP109_play", "cast": "IMP109", "text": "[giddy] Dibs on everything!"}, {"stem": "IMP109_attack", "cast": "IMP109", "text": "[cheeky] Finders keepers!"}, {"stem": "IMP109_death", "cast": "IMP109", "text": "[heartbroken] My treasure..."}, {"stem": "IMP110_play", "cast": "IMP110", "text": "[pompous] Behold, magnificence!"}, {"stem": "IMP110_attack", "cast": "IMP110", "text": "[arrogant] Witness me!"}, {"stem": "IMP110_death", "cast": "IMP110", "text": "[dramatic] Remember my name..."}, {"stem": "IMP111_play", "cast": "IMP111", "text": "[contemptuous] Bow, insects."}, {"stem": "IMP111_attack", "cast": "IMP111", "text": "[roaring] Burn, all of you!"}, {"stem": "IMP111_death", "cast": "IMP111", "text": "[mournful] The sky weeps..."}, {"stem": "IMP113_play", "cast": "IMP113", "text": "[roaring] Wyrmscar awakens!"}, {"stem": "IMP113_attack", "cast": "IMP113", "text": "[roaring] Burn to cinders!"}, {"stem": "IMP114_play", "cast": "IMP114", "text": "[steadfast] Always ready."}, {"stem": "IMP114_attack", "cast": "IMP114", "text": "[battle cry] Spear of embers!"}, {"stem": "IMP114_death", "cast": "IMP114", "text": "[solemn] My watch ends..."}, {"stem": "IMP127_play", "cast": "IMP127", "text": "[calm] Face me with honor."}, {"stem": "IMP127_attack", "cast": "IMP127", "text": "[sharp] One cut."}, {"stem": "IMP127_death", "cast": "IMP127", "text": "[peaceful] Blossoms... fall..."}, {"stem": "IMP128_play", "cast": "IMP128", "text": "[whispers] You never looked up."}, {"stem": "IMP128_attack", "cast": "IMP128", "text": "[playful] Surprise!"}, {"stem": "IMP128_death", "cast": "IMP128", "text": "[weak] Fallen..."}, {"stem": "IMP129_play", "cast": "IMP129", "text": "[resolute] The roots will hold."}, {"stem": "IMP129_attack", "cast": "IMP129", "text": "[battle cry] For the Crown!"}, {"stem": "IMP129_death", "cast": "IMP129", "text": "[fading] The roots endure..."}, {"stem": "IMP130_play", "cast": "IMP130", "text": "[booming] The wild answers!"}, {"stem": "IMP130_attack", "cast": "IMP130", "text": "[booming] Return to the earth!"}, {"stem": "IMP130_death", "cast": "IMP130", "text": "[rumbling] To the soil..."}, {"stem": "IMP131_play", "cast": "IMP131", "text": "[mystical] Spirits, guide me."}, {"stem": "IMP131_attack", "cast": "IMP131", "text": "[ominous] Nature's wrath!"}, {"stem": "IMP131_death", "cast": "IMP131", "text": "[peaceful] Back to the grove..."}, {"stem": "IMP132_play", "cast": "IMP132", "text": "[eerie whisper] Feed the roots..."}, {"stem": "IMP132_death", "cast": "IMP132", "text": "[hissing whisper] Wither..."}, {"stem": "IMP134_play", "cast": "IMP134", "text": "[teasing, playful] Shall we dance?"}, {"stem": "IMP134_attack", "cast": "IMP134", "text": "[amused laugh] Too slow, darling."}, {"stem": "IMP134_death", "cast": "IMP134", "text": "[soft, pained] My tails dim..."}, {"stem": "IMP135_play", "cast": "IMP135", "text": "[smug] Never saw me coming."}, {"stem": "IMP135_attack", "cast": "IMP135", "text": "[whispers] Behind you."}, {"stem": "IMP135_death", "cast": "IMP135", "text": "[fading] Unseen..."}, {"stem": "IMP136_play", "cast": "IMP136", "text": "[dreamy] The moon whispers..."}, {"stem": "IMP136_attack", "cast": "IMP136", "text": "[quiet] Strike, then vanish."}, {"stem": "IMP136_death", "cast": "IMP136", "text": "[soft] The moon sets..."}, {"stem": "IMP138_play", "cast": "IMP138", "text": "[focused] Bowstring's taut."}, {"stem": "IMP138_attack", "cast": "IMP138", "text": "[sharp] Loose! Again!"}, {"stem": "IMP138_death", "cast": "IMP138", "text": "[pained] Out of arrows..."}, {"stem": "IMP139_play", "cast": "IMP139", "text": "[rallying] Sylvans, to me!"}, {"stem": "IMP139_attack", "cast": "IMP139", "text": "[energetic] Rally and strike!"}, {"stem": "IMP139_death", "cast": "IMP139", "text": "[urgent] Keep the call..."}, {"stem": "IMP140_play", "cast": "IMP140", "text": "[eager] I'll do my best!"}, {"stem": "IMP140_attack", "cast": "IMP140", "text": "[effort shout] Hyah! Like I practiced!"}, {"stem": "IMP140_death", "cast": "IMP140", "text": "[regretful] Should've trained more..."}, {"stem": "IMP141_play", "cast": "IMP141", "text": "[gentle] Walk softly here."}, {"stem": "IMP141_attack", "cast": "IMP141", "text": "[firm] Leave our home."}, {"stem": "IMP141_death", "cast": "IMP141", "text": "[sorrowful] Forgive me, forest..."}, {"stem": "IMP142_play", "cast": "IMP142", "text": "[playful] Catch me if you can!"}, {"stem": "IMP142_attack", "cast": "IMP142", "text": "[laughing] Over here!"}, {"stem": "IMP142_death", "cast": "IMP142", "text": "[sheepish] Okay, caught..."}, {"stem": "IMP143_play", "cast": "IMP143", "text": "[eager] Little fox, big help!"}, {"stem": "IMP143_attack", "cast": "IMP143", "text": "[spirited] Take that!"}, {"stem": "IMP143_death", "cast": "IMP143", "text": "[proud] I did my part..."}, {"stem": "IMP144_play", "cast": "IMP144", "text": "[mischievous] Dance, puppet."}, {"stem": "IMP144_attack", "cast": "IMP144", "text": "[teasing] Was that real?"}, {"stem": "IMP144_death", "cast": "IMP144", "text": "[fading laugh] Just an illusion..."}, {"stem": "IMP163_play", "cast": "IMP163", "text": "[quiet] One arrow is enough."}, {"stem": "IMP163_attack", "cast": "IMP163", "text": "[whispers] Right between the plates."}, {"stem": "IMP163_death", "cast": "IMP163", "text": "[fading] Into the shadow..."}, {"stem": "IMP160_play", "cast": "IMP160", "text": "[cheerful] Special delivery!"}, {"stem": "IMP160_attack", "cast": "IMP160", "text": "[startled] Return to sender!"}, {"stem": "IMP160_death", "cast": "IMP160", "text": "[sighs] Delivery late..."}, {"stem": "IMP034_play", "cast": "IMP034", "text": "[synthetic] Guardian online."}, {"stem": "IMP034_attack", "cast": "IMP034", "text": "[flat] Engaging target."}, {"stem": "IMP034_death", "cast": "IMP034", "text": "[glitching] Shutting down..."}, {"stem": "IMP035_play", "cast": "IMP035", "text": "[over the radio] Argentum rolling!"}, {"stem": "IMP035_attack", "cast": "IMP035", "text": "[shouting] Main cannon, fire!"}, {"stem": "IMP035_death", "cast": "IMP035", "text": "[panicked] Abandon the tank!"}, {"stem": "IMP036_play", "cast": "IMP036", "text": "[over the radio] Carrier deployed."}, {"stem": "IMP036_attack", "cast": "IMP036", "text": "[shouting] Ramming speed!"}, {"stem": "IMP036_death", "cast": "IMP036", "text": "[urgent] Drones, deploy!"}, {"stem": "IMP049_play", "cast": "IMP049", "text": "[over the radio] Locked and loaded."}, {"stem": "IMP049_attack", "cast": "IMP049", "text": "[shouting] Fire for effect!"}, {"stem": "IMP049_death", "cast": "IMP049", "text": "[coughing] We're out..."}, {"stem": "IMP050_play", "cast": "IMP050", "text": "[synthetic] All systems green."}, {"stem": "IMP050_attack", "cast": "IMP050", "text": "[cold, synthetic] Stomp protocol."}, {"stem": "IMP050_death", "cast": "IMP050", "text": "[glitching] Critical failure..."}, {"stem": "IMP065_play", "cast": "IMP065", "text": "[synthetic] Package ready."}, {"stem": "IMP065_attack", "cast": "IMP065", "text": "[synthetic] Delivery incoming."}, {"stem": "IMP065_death", "cast": "IMP065", "text": "[synthetic] Signing off."}, {"stem": "IMP067_play", "cast": "IMP067", "text": "[authoritative] Airspace locked."}, {"stem": "IMP067_attack", "cast": "IMP067", "text": "[commanding] All batteries, fire!"}, {"stem": "IMP067_death", "cast": "IMP067", "text": "[panicked] Abandon ship!"}, {"stem": "IMP068_play", "cast": "IMP068", "text": "[synthetic] Scanning..."}, {"stem": "IMP068_attack", "cast": "IMP068", "text": "[cold, synthetic] Heretic detected."}, {"stem": "IMP068_death", "cast": "IMP068", "text": "[glitching] Scan terminated..."}, {"stem": "IMP076_play", "cast": "IMP076", "text": "[cocky] Weapons hot!"}, {"stem": "IMP076_attack", "cast": "IMP076", "text": "[excited] Strafing run!"}, {"stem": "IMP076_death", "cast": "IMP076", "text": "[panicked] Mayday! Mayday!"}, {"stem": "IMP079_play", "cast": "IMP079", "text": "[cold] All targets acquired."}, {"stem": "IMP079_attack", "cast": "IMP079", "text": "[menacing] Annihilation cannon!"}, {"stem": "IMP079_death", "cast": "IMP079", "text": "[alarmed] Reactor critical..."}, {"stem": "IMP081_play", "cast": "IMP081", "text": "[friendly] Cargo coming home."}, {"stem": "IMP081_attack", "cast": "IMP081", "text": "[shouting] Shields forward!"}, {"stem": "IMP081_death", "cast": "IMP081", "text": "[dismayed] Cargo lost..."}, {"stem": "IMP159_play", "cast": "IMP159", "text": "[gruff] Let's dig!"}, {"stem": "IMP159_attack", "cast": "IMP159", "text": "[shouting] Stand back!"}, {"stem": "IMP159_death", "cast": "IMP159", "text": "[sputtering] Engine stalled..."}, {"stem": "TOKEN_DRONE_play", "cast": "TOKEN_DRONE", "text": "[synthetic] Drone active."}, {"stem": "TOKEN_DRONE_attack", "cast": "TOKEN_DRONE", "text": "[synthetic] Engaging."}, {"stem": "TOKEN_DRONE_death", "cast": "TOKEN_DRONE", "text": "[glitching] Signal lost."}, {"stem": "IMP001_play", "cast": "IMP001", "text": "[guttural] Assimilate..."}, {"stem": "IMP001_attack", "cast": "IMP001", "text": "[guttural roar] Feed the machine!"}, {"stem": "IMP001_death", "cast": "IMP001", "text": "[dying groan] Failing..."}, {"stem": "IMP074_play", "cast": "IMP074", "text": "[guttural, labored] Stitched together..."}, {"stem": "TOKEN_EXPERIMENT_play", "cast": "TOKEN_EXPERIMENT", "text": "[guttural] Obey, master..."}, {"stem": "ANN_yourTurn", "cast": "ANN", "text": "[confident] Your move, Sovereign."}, {"stem": "ANN_victory", "cast": "ANN", "text": "[triumphant] Victory!"}, {"stem": "ANN_defeat", "cast": "ANN", "text": "[somber] Defeat..."}, {"stem": "ANN_begin", "cast": "ANN", "text": "[dramatic] The battle begins!"}, {"stem": "ANN_zoneWon", "cast": "ANN", "text": "[excited] Zone secured!"}], "redo": ["IMP041", "IMP017", "IMP104", "IMP044", "IMP014"], "redo_extra": ["IMP011", "IMP073", "IMP046", "IMP078", "IMP134"], "retire": ["paladin"]}""")
CORE = set(BIBLE.get('redo') or [])
REDO = set(CORE)
if REDO and not SCOPE.startswith('Only'): REDO |= set(BIBLE.get('redo_extra') or [])
JOBS = [j for j in BIBLE['jobs'] if not REDO or j['stem'].rsplit('_', 1)[0] in REDO]
JOBS.sort(key=lambda j: j['stem'].rsplit('_', 1)[0] not in CORE)   # reported characters first
NEEDED = {BIBLE['cast'][j['cast']]['design'] for j in JOBS if BIBLE['cast'][j['cast']]['design']}
if REDO: print('Re-recording round for:', ', '.join(sorted(REDO, key=lambda c: (c not in CORE, c))))
ROUND = ('_redo_' + hashlib.sha1(json.dumps(JOBS, sort_keys=True).encode()).hexdigest()[:6]) if REDO else ''  # each round records into its own folder
TOTAL = sum(len(j['text']) for j in JOBS)
PREMADE = [v for v in VOICES if v.get('category') == 'premade']
BY_NAME = {}
for v in PREMADE: BY_NAME.setdefault(v['name'].split(' - ')[0].strip(), v)
MINE = {v['name']: v for v in VOICES if v['name'].startswith('EoA ')}
limit = SUB.get('voice_limit') or 10
if SUB.get('voice_slots_used') is not None:   # the account's own count already includes the EoA voices
    FREE = max(0, limit - SUB['voice_slots_used'])
else:
    used = sum(1 for v in VOICES if v.get('category') in ('generated', 'cloned') and not v['name'].startswith('EoA '))
    FREE = max(0, limit - used - len(MINE))
left = (SUB.get('character_limit', 30000) - SUB.get('character_count', 0)) if SUB else 30000
missing = [d for d in BIBLE['designs'] if d['key'] in NEEDED and 'EoA ' + d['key'] not in MINE]
n_new = len(missing) if REDO else min(FREE, len(missing))
design_cost = n_new * 600
if REDO: TAKES = [0.0, 0.0, 0.0, 0.5]   # few lines: several expressive takes to choose from
else: TAKES = [0.5, 0.0] if left >= 2.15 * TOTAL + design_cost else [0.5]
print(f'Clips: {len(JOBS)} · characters per take: {TOTAL:,}')
print(f'Free voice slots: {FREE} (EoA voices already on the account: {len(MINE)}) → new designed voices: {n_new}')
print(f'Takes per line: {len(TAKES)}; estimated cost ≈ {len(TAKES) * TOTAL + design_cost:,} credits of {left:,} available')
if left < TOTAL + design_cost:
    print('WARNING: not enough credits for everything. Step 5 will record as much as it can; finish the rest next month or after topping up.')


In [ ]:
#@title 3) Design unique voices (listen to them)
for key in (BIBLE.get('retire') or []):
    v = MINE.pop('EoA ' + key, None)
    if v:
        r = call('DELETE', f"/v1/voices/{v['voice_id']}")
        print(('✓ removed old voice ' if r.status_code == 200 else f'✗ {r.status_code} ') + 'EoA ' + key)
        if r.status_code == 200: FREE += 1
DESIGNED = {v['name'][4:]: v['voice_id'] for v in MINE.values()}
todo = [d for d in BIBLE['designs'] if d['key'] not in DESIGNED and d['key'] in NEEDED]
if REDO and FREE < len(todo):
    # free slots by removing designed voices this round does not need (each can be recreated identically from its seed)
    order = {d['key']: i for i, d in enumerate(BIBLE['designs'])}
    spare = sorted([n for n in MINE if n[4:] not in NEEDED], key=lambda n: -order.get(n[4:], 999))  # least central designs first
    for name in spare[:len(todo) - FREE]:
        r = call('DELETE', f"/v1/voices/{MINE[name]['voice_id']}")
        if r.status_code == 200:
            print(f'– freed a slot: removed {name} (it can be recreated later from its seed)')
            DESIGNED.pop(name[4:], None); MINE.pop(name); FREE += 1
free = FREE
for d in todo:
    key = d['key']
    if free <= 0:
        print(f'– {key}: no free slot, a default voice will be used'); continue
    body = {'voice_description': d['description'], 'model_id': 'eleven_ttv_v3', 'text': d['text'], 'seed': d['seed'], 'guidance_scale': d.get('guidance', 5.0)}
    r = call('POST', '/v1/text-to-voice/design', json=body)
    if r.status_code != 200:
        body.pop('model_id'); r = call('POST', '/v1/text-to-voice/design', json=body)
    if r.status_code != 200:
        print(f'✗ {key}: design failed ({r.status_code}) {err(r)}'); continue
    pv = r.json()['previews'][0]
    cb = {'voice_name': 'EoA ' + key, 'voice_description': d['description'], 'generated_voice_id': pv['generated_voice_id']}
    c = call('POST', '/v1/text-to-voice', json=cb)
    if c.status_code in (404, 405):
        c = call('POST', '/v1/text-to-voice/create-voice-from-preview', json=cb)
    if c.status_code != 200:
        print(f'✗ {key}: could not save the voice ({c.status_code}) {err(c)}'); continue
    DESIGNED[key] = c.json()['voice_id']; free -= 1
    player(base64.b64decode(pv['audio_base_64']), pv.get('media_type') or 'audio/mpeg', f'{key}: {d["description"][:90]}…')
print(f'Designed voices ready: {len(DESIGNED)}')


In [ ]:
#@title 4) Cast and a short test
def resolve(key):
    c = BIBLE['cast'][key]
    if c['design'] and c['design'] in DESIGNED: return DESIGNED[c['design']], 'EoA ' + c['design']
    for n in c['premade']:
        if n in BY_NAME: return BY_NAME[n]['voice_id'], n
    for v in PREMADE:
        if (v.get('labels') or {}).get('gender') == c['gender']: return v['voice_id'], v['name']
    v = (PREMADE or VOICES)[0]; return v['voice_id'], v['name']
CASTMAP = {k: resolve(k) for k in BIBLE['cast']}
FORMAT = 'mp3_44100_64'

def tts(voice_id, text, stability):
    global FORMAT
    body = {'text': text, 'model_id': 'eleven_v3', 'voice_settings': {'stability': stability, 'similarity_boost': 0.75}}
    r = call('POST', f'/v1/text-to-speech/{voice_id}?output_format={FORMAT}', json=body)
    if r.status_code in (400, 422) and 'output_format' in r.text:
        FORMAT = 'mp3_44100_128'; r = call('POST', f'/v1/text-to-speech/{voice_id}?output_format={FORMAT}', json=body)
    if r.status_code in (400, 422):
        r = call('POST', f'/v1/text-to-speech/{voice_id}?output_format={FORMAT}', json={'text': text, 'model_id': 'eleven_v3'})
    if r.status_code != 200: raise RuntimeError(f'{r.status_code}: {err(r)}')
    return r.content

roles = {}
for k, (vid, name) in CASTMAP.items():
    if not REDO or k in REDO: roles.setdefault(name, []).append(k)
print('Cast (voice → roles):')
for name, ks in sorted(roles.items()): print(f'  {name}: {", ".join(ks)}')
by_stem = {j['stem']: j for j in JOBS}
for stem in ([j['stem'] for j in JOBS if j['stem'].endswith('_play')][:10] if REDO else ['IMP015_play', 'IMP003_play', 'IMP111_play', 'IMP051_play']):
    j = by_stem[stem]
    try:
        player(tts(CASTMAP[j['cast']][0], j['text'], 0.0 if REDO else 0.5), 'audio/mpeg', f'{stem} ({CASTMAP[j["cast"]][1]}): {j["text"]}')
    except Exception as e:
        print('Test error:', stem, e)


In [ ]:
#@title 5) Record everything (you can run it again: it continues where it stopped)
OUTD = '/content/eoa_voices' + ROUND; os.makedirs(OUTD, exist_ok=True)
todo = [(j, k, st) for j in JOBS for k, st in enumerate(TAKES)]
todo = [t for t in todo if not (os.path.exists(f"{OUTD}/{t[0]['stem']}__t{t[1]}.mp3") and os.path.getsize(f"{OUTD}/{t[0]['stem']}__t{t[1]}.mp3") > 800)]
print(f'To record: {len(todo)} clips')
lock = threading.Lock(); done = [0]; fails = []; t0 = time.time(); stop = [False]

def work(item):
    j, k, st = item
    if stop[0]: return
    try:
        audio = tts(CASTMAP[j['cast']][0], j['text'], st)
        with open(f"{OUTD}/{j['stem']}__t{k}.mp3", 'wb') as f: f.write(audio)
    except Exception as e:
        with lock: fails.append((j['stem'], k, str(e)[:200]))
        if 'quota' in str(e).lower() or 'credits' in str(e).lower(): stop[0] = True
    with lock:
        done[0] += 1
        if done[0] % 20 == 0 or done[0] == len(todo):
            el = time.time() - t0; eta = el / done[0] * (len(todo) - done[0])
            print(f'{done[0]}/{len(todo)} · {el / 60:.1f} min · about {eta / 60:.0f} min left', flush=True)

with cf.ThreadPoolExecutor(max_workers=2) as ex: list(ex.map(work, todo))
json.dump({'cast': {k: {'voice_id': v[0], 'voice': v[1]} for k, v in CASTMAP.items()}, 'takes': TAKES, 'format': FORMAT, 'jobs': JOBS},
          open(f'{OUTD}/manifest.json', 'w'), ensure_ascii=False, indent=1)
print(f'Done. Errors: {len(fails)}')
for f in fails[:15]: print('  ', f)
if stop[0]: print('Recording stopped: credits ran out. Top up or wait for the reset, then run step 5 again.')
elif fails: print('Run step 5 again to retry the failed clips.')


In [ ]:
#@title 6) Pack into ZIP and download
from google.colab import files
names = sorted(n for n in os.listdir(OUTD) if n.endswith('.mp3'))
parts, cur, size = [], [], 0
for n in names:
    s = os.path.getsize(f'{OUTD}/{n}')
    if cur and size + s > 22_000_000: parts.append(cur); cur, size = [], 0
    cur.append(n); size += s
if cur: parts.append(cur)
for i, p in enumerate(parts, 1):
    zp = f'/content/eoa_voices{"_redo" if REDO else ""}_part{i}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as z:
        if i == 1: z.write(f'{OUTD}/manifest.json', 'manifest.json')
        for n in p: z.write(f'{OUTD}/{n}', n)
    print(f'Part {i}: {len(p)} files, {os.path.getsize(zp) / 1e6:.1f} MB')
    files.download(zp)
print('If the browser asks whether to allow multiple downloads, choose Allow. Send all the parts to Claude.')


In [ ]:
#@title 7) (optional) Remove the designed voices from your account
mine = [v for v in call('GET', '/v1/voices').json().get('voices', []) if v['name'].startswith('EoA ')]
for v in mine:
    r = call('DELETE', f"/v1/voices/{v['voice_id']}")
    print(('✓ removed ' if r.status_code == 200 else f'✗ {r.status_code} ') + v['name'])
print('Done. The recorded clips are not affected.')
